# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [1]:
!pip install langchain_community langchain_experimental langchain_openai

INFO: pip is looking at multiple versions of langchain-experimental to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-experimental to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.4/199.4 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: langchain_experimental
    Found existing installation: langchain-experimental 0.4.1
    Uninstalling langchain-experimental-0.4.1:
      Successfully uninstalled langchain-experimental-0.4.1


In [2]:
from typing import Optional

from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [7]:
%pip install faiss-cpu > /dev/null
%pip install google-search-results > /dev/null
from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [4]:
import os
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
SERPAPI_API_KEY = userdata.get('SERPAPI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["SERPAPI_API_KEY"] = SERPAPI_API_KEY

In [8]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings()
# Initialize the vectorstore as empty
import faiss

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [14]:
from langchain.agents import AgentExecutor, Tool, ZeroShotAgent
from langchain.chains import LLMChain
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import OpenAI

todo_prompt = PromptTemplate.from_template(
    """
    You are an expert AI project planner.

    Create a short and practical task list for the following objective:

    {objective}

    Rules:
    - Maximum 6 tasks
    - Tasks must be actionable
    - Include research, planning, implementation, and evaluation
    - Avoid repetitive or generic tasks
    """
)

todo_chain = LLMChain(
    llm=OpenAI(temperature=0),
    prompt=todo_prompt
)

search = SerpAPIWrapper()

tools = [
    Tool(
        name="Search",
        func=search.run,
        description="Useful for researching technologies, frameworks, trends, and current information"
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="Useful for generating structured project task lists from an objective"
    ),
]

prefix = """
You are an AI engineering assistant working toward this objective:

{objective}

Previously completed tasks:
{context}

Rules:
- Think step-by-step
- Use tools when needed
- Produce useful engineering and planning outputs
- Avoid repeating todo lists endlessly
- Focus on practical results
"""

suffix = """
Question: {task}
{agent_scratchpad}
"""

prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=[
        "objective",
        "task",
        "context",
        "agent_scratchpad"
    ],
)

In [10]:
llm = OpenAI(temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `ZeroShotAgent` was deprecated in LangChain 0.1.0 and will be removed in 0.3.0. Use create_react_agent instead.
  warn_deprecated(


### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [26]:
import faiss
from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)

vectorstore = FAISS(
    embeddings_model.embed_query,
    index,
    InMemoryDocstore({}),
    {}
)

In [27]:
OBJECTIVE = "Create an AI-powered coding mentor for beginner programmers that analyzes their code, explains mistakes, gives personalized exercises, and generates a complete MVP launch strategy."

In [28]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [29]:
baby_agi({"objective": OBJECTIVE})


*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
Thought: I should ask the user what they want to create a todo list for
Action: TODO
Action Input: Objective to create a todo list for
Observation:  a successful job search

1. Research and identify potential job opportunities in your desired field or industry.
2. Update and tailor your resume and cover letter to fit each job application.
3. Network and reach out to contacts in your industry for potential job leads or referrals.
4. Utilize online job search engines and company websites to search for job openings.
5. Attend job fairs and career events to connect with potential employers.
6. Prepare for interviews by researching the company, practicing common interview questions, and dressing professionally.
7. Follow up with employers after interviews to express your interest and inquire about next steps.
8. Consider expanding your job search to include different l

{'objective': 'Create an AI-powered coding mentor for beginner programmers that analyzes their code, explains mistakes, gives personalized exercises, and generates a complete MVP launch strategy.'}